## 1. Context

The purpose of this notebook is to clean and prepare a dataset of sales from **Amazon India** so that it can be used for analysis.

The dataset comes from **Kaggle – Amazon Sale Report** and contains information about orders placed on Amazon India, including products, quantities sold, order amounts, dates, and shipping locations.

The main steps include:

- dataset exploration
- handling missing values
- converting columns to the appropriate data types
- creating useful variables for analysis
- exporting cleaned datasets

The cleaned datasets will then be used to perform sales analysis and build an Excel dashboard.

## 2. Importing Libraries

In [1]:
import pandas as pd

## 3. Data Loading

The dataset contains information about sales made on Amazon, including products, quantities sold, amounts, order dates, and geographic locations.

In [2]:
df = pd.read_csv(
    '../data/raw/Amazon Sale Report.csv',
    encoding ='UTF8',
    low_memory=False)

# Display all columns when exploring the dataset
pd.set_option("display.max_columns", None)

## 4. Dataset Exploration

This step aims to identify the structure of the data as well as any potential missing values.

### Dataset Overview

In [3]:
df.head()

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,ASIN,Courier Status,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


### Dataset Structure

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  object 
 2   Date                128975 non-null  object 
 3   Status              128975 non-null  object 
 4   Fulfilment          128975 non-null  object 
 5   Sales Channel       128975 non-null  object 
 6   ship-service-level  128975 non-null  object 
 7   Style               128975 non-null  object 
 8   SKU                 128975 non-null  object 
 9   Category            128975 non-null  object 
 10  Size                128975 non-null  object 
 11  ASIN                128975 non-null  object 
 12  Courier Status      122103 non-null  object 
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  object 
 15  Amount              121180 non-nul

### Statistical Summary

In [5]:
df.describe()

,index,Qty,Amount,ship-postal-code
count,128975.000000,128975.000000,121180.000000,128942.000000
mean,64487.000000,0.904431,648.561465,463966.236509
std,37232.019822,0.313354,281.211687,191476.764941
min,0.000000,0.000000,0.000000,110001.000000
25%,32243.500000,1.000000,449.000000,382421.000000
50%,64487.000000,1.000000,605.000000,500033.000000
75%,96730.500000,1.000000,788.000000,600024.000000
max,128974.000000,15.000000,5584.000000,989898.000000


### Duplicate Check

This step verifies whether the dataset contains duplicate rows.

In [6]:
df.duplicated().sum()

np.int64(0)

No duplicates were detected in the dataset.

### Column Analysis

This step provides an overview of the dataset’s columns, including their data types, number of unique values, percentage of missing values, and examples of values.

In [7]:
analysis = []
for col in df.columns:
    try:
        col_type = df[col].dtype
        unique_values = df[col].dropna().unique()[:5].tolist()
        analysis.append({
            'Column Name': col
            , 'Column Type': str(col_type) 
            , 'N Unique': df[col].nunique()
            , 'Null %': f"{df[col].isnull().mean():.1%}"
            , 'Sample Values': str(unique_values)[:100]
        })
    except Exception as e:
        analysis.append({
            'Column Name': col
            , 'Column Type': 'ERROR'
            , 'N Unique': 'N/A'
            , 'Null %': 'N/A'
            , 'Sample Values': str(e)
        })
pd.DataFrame(analysis)

,Column Name,Column Type,N Unique,Null %,Sample Values
0,index,int64,128975,0.0%,"[0, 1, 2, 3, 4]"
1,Order ID,object,120378,0.0%,"['405-8078784-5731545', '171-9198151-1101146',..."
2,Date,object,91,0.0%,"['04-30-22', '04-29-22', '04-28-22', '04-27-22..."
3,Status,object,13,0.0%,"['Cancelled', 'Shipped - Delivered to Buyer', ..."
4,Fulfilment,object,2,0.0%,"['Merchant', 'Amazon']"
5,Sales Channel,object,2,0.0%,"['Amazon.in', 'Non-Amazon']"
6,ship-service-level,object,2,0.0%,"['Standard', 'Expedited']"
7,Style,object,1377,0.0%,"['SET389', 'JNE3781', 'JNE3371', 'J0341', 'JNE..."
8,SKU,object,7195,0.0%,"['SET389-KR-NP-S', 'JNE3781-KR-XXXL', 'JNE3371..."
9,Category,object,9,0.0%,"['Set', 'kurta', 'Western Dress', 'Top', 'Ethn..."


### Missing Values Analysis

This step aims to identify the columns containing missing values and determine their proportion within the dataset.

In [8]:
missing_pct = df.isnull().mean() * 100

result = pd.DataFrame({
    'column': missing_pct.index,
    'missing_pct': missing_pct.values.round(2)
})

result = (
    result
    .query('missing_pct > 0')
    .sort_values('missing_pct', ascending=False)
    .reset_index(drop=True)
)

result

,column,missing_pct
0,fulfilled-by,69.55
1,promotion-ids,38.11
2,Unnamed: 22,38.03
3,currency,6.04
4,Amount,6.04
5,Courier Status,5.33
6,ship-postal-code,0.03
7,ship-state,0.03
8,ship-city,0.03
9,ship-country,0.03


We observe that the columns **'fulfilled-by'**, **'promotion-ids'**, and **'Unnamed: 22'** contain a high proportion of missing values.  
These columns will be examined during the data cleaning step.

## 5. Data Cleaning

In [9]:
df_wip = df.copy()

### Column Name Cleaning

In [10]:
df_wip.columns = df_wip.columns.str.strip()

### Removal of Irrelevant Columns

In [11]:
df_wip["Unnamed: 22"].value_counts(dropna=False)

Unnamed: 22
False    79925
NaN      49050
Name: count, dtype: int64

In [12]:
df_wip["currency"].value_counts(dropna=False)

currency
INR    121180
NaN      7795
Name: count, dtype: int64

In [13]:
df_wip["fulfilled-by"].value_counts(dropna=False)

fulfilled-by
NaN          89698
Easy Ship    39277
Name: count, dtype: int64

In [14]:
df_wip["Sales Channel"].value_counts(dropna=False)

Sales Channel
Amazon.in     128851
Non-Amazon       124
Name: count, dtype: int64

In [15]:
df_wip['ship-country'].value_counts(dropna=False)

ship-country
IN     128942
NaN        33
Name: count, dtype: int64

In [16]:
df_wip = df_wip.drop(columns=[
    "Unnamed: 22",
    "index",
    "currency",
    "fulfilled-by",
    "Sales Channel",
    "ship-country"
])

In [17]:
df_wip.columns

Index(['Order ID', 'Date', 'Status', 'Fulfilment', 'ship-service-level',
       'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty',
       'Amount', 'ship-city', 'ship-state', 'ship-postal-code',
       'promotion-ids', 'B2B'],
      dtype='object')

The following columns were removed because they do not provide relevant information for the analysis:

- **Unnamed: 22**: technical column generated during the dataset import, containing mostly missing values.
- **index**: redundant column duplicating the dataframe index.
- **currency**: contains only the value **INR** and therefore provides no variability.
- **fulfilled-by**: contains a large proportion of missing values and is not useful for sales analysis.
- **Sales Channel**: mostly contains the value **"Amazon.in"** with very few **"Non-Amazon"** observations, which limits its analytical usefulness in this project.
- **ship-country**: contains almost exclusively the value **"IN"** (India), meaning the dataset only covers sales from Amazon India. This variable therefore provides no additional analytical value.

### Standardization of categorical values

Some categorical variables contain inconsistencies in capitalization and extra spaces.  
To ensure consistency and avoid duplicate categories during analysis, the columns 'Category' and 'ship-city' are standardized by trimming whitespace and applying consistent capitalization.

In [18]:
df_wip["Category"].value_counts()

Category
Set              50284
kurta            49877
Western Dress    15500
Top              10622
Ethnic Dress      1159
Blouse             926
Bottom             440
Saree              164
Dupatta              3
Name: count, dtype: int64

In [19]:
df_wip["ship-city"].value_counts()

ship-city
BENGALURU                                             11217
HYDERABAD                                              8074
MUMBAI                                                 6126
NEW DELHI                                              5795
CHENNAI                                                5421
                                                      ...  
Harwada, Ankola                                           1
Titwala ( East)                                           1
PATNApatna                                                1
Sbi Fagu                                                  1
Goppenahalli, Channagiri Taluk, Davangere district        1
Name: count, Length: 8955, dtype: int64

The 'ship-city' column contains several inconsistencies such as differences in capitalization, additional location details, parentheses, punctuation, and extra spaces.

To ensure consistency in geographic analysis, a series of standardization steps are applied:

- Removal of additional location details contained in parentheses.
- When multiple locations are present, only the first one is retained.
- Removal of trailing punctuation.
- Standardization of capitalization.
- Cleaning of leading/trailing and duplicate spaces.

These transformations help avoid artificial duplication of cities in the analysis ("Chennai", "CHENNAI", "Chennai (East)"...).

In [20]:
# Standardization of city names

# Remove information contained in parentheses
df_wip["ship-city"] = df_wip["ship-city"].str.replace(r"\(.*?\)", "", regex=True)

# If multiple locations are listed, keep only the first one
df_wip["ship-city"] = df_wip["ship-city"].str.split(",").str[0]

# Remove trailing periods at the end of city names
df_wip["ship-city"] = df_wip["ship-city"].str.replace(r"\.$", "", regex=True)

# Trim leading/trailing spaces and standardize capitalization
df_wip["ship-city"] = df_wip["ship-city"].str.strip().str.title()

# Remove multiple consecutive spaces
df_wip["ship-city"] = df_wip["ship-city"].str.replace(r"\s+", " ", regex=True)

# Verification of the cleaned values
df_wip["ship-city"].value_counts()

ship-city
Bengaluru                 11902
Hyderabad                  9144
Mumbai                     7135
New Delhi                  6343
Chennai                    6288
                          ...  
Avadi                         1
Pudupalayam Agraharam         1
Tirva Rod Gursahaiganj        1
Moodabettu                    1
Ayyampalayam                  1
Name: count, Length: 6740, dtype: int64

In [21]:
df_wip["Category"] = df_wip["Category"].str.strip().str.title()
df_wip["ship-city"] = df_wip["ship-city"].str.strip().str.title()

### Creating a New Feature

In [22]:
df_wip["promotion_used"] = df_wip["promotion-ids"].notna()
df_wip = df_wip.drop(columns=["promotion-ids"])

The **'promotion-ids'** column contains the identifier of a promotion when a promotion was applied to an order.

To simplify the analysis, a new boolean variable **'promotion_used'** is created:
- **True** if a promotion was used
- **False** otherwise

The **'promotion-ids'** column is then removed since the useful information is now contained in the new variable.

### Data Type Conversion

In [23]:
df_wip["Date"] = pd.to_datetime(df_wip["Date"],format="%m-%d-%y")
df_wip["Amount"] = pd.to_numeric(df_wip["Amount"])
df_wip["ship-postal-code"] = df_wip["ship-postal-code"].astype("Int64").astype("string")

In [24]:
df_wip.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Order ID            128975 non-null  object        
 1   Date                128975 non-null  datetime64[ns]
 2   Status              128975 non-null  object        
 3   Fulfilment          128975 non-null  object        
 4   ship-service-level  128975 non-null  object        
 5   Style               128975 non-null  object        
 6   SKU                 128975 non-null  object        
 7   Category            128975 non-null  object        
 8   Size                128975 non-null  object        
 9   ASIN                128975 non-null  object        
 10  Courier Status      122103 non-null  object        
 11  Qty                 128975 non-null  int64         
 12  Amount              121180 non-null  float64       
 13  ship-city           128942 no

### Analysis of Missing Values and Zero Amounts

Rows where the **'Amount'** column is missing are examined in order to understand in which situations this occurs.

In [25]:
df_wip[df_wip["Amount"].isna()]["Status"].value_counts()

Status
Cancelled                       7566
Shipped                          208
Shipped - Delivered to Buyer       8
Shipping                           8
Shipped - Returned to Seller       3
Pending                            2
Name: count, dtype: int64

We observe that the majority of these rows correspond to cancelled orders.  
In these cases, no amount is recorded because the transaction was not completed.

We then examine orders for which the amount is equal to zero.

In [26]:
df_wip[df_wip["Amount"] == 0]["Status"].value_counts()

Status
Shipped                          1518
Shipped - Delivered to Buyer      716
Shipped - Returned to Seller       51
Shipped - Picked Up                28
Pending                            17
Pending - Waiting for Pick Up       9
Shipped - Lost in Transit           2
Shipped - Returning to Seller       2
Name: count, dtype: int64

These cases mainly correspond to cancelled, returned, or uncompleted orders.  
These transactions therefore do not represent actual sales.

Rows with a missing amount are removed because they do not correspond to valid transactions.

In [27]:
df_wip = df_wip.dropna(subset=["Amount"])

We also check the rows where the quantity sold is equal to zero.

In [28]:
df_wip[df_wip["Qty"] == 0]["Status"].value_counts()

Status
Cancelled    5136
Name: count, dtype: int64

These rows also correspond to cancelled orders and do not represent actual sales.  
They will therefore be excluded from the dataset used for sales analysis.

In [29]:
df_wip["Status"].value_counts()

Status
Shipped                          77596
Shipped - Delivered to Buyer     28761
Cancelled                        10766
Shipped - Returned to Seller      1950
Shipped - Picked Up                973
Pending                            656
Pending - Waiting for Pick Up      281
Shipped - Returning to Seller      145
Shipped - Out for Delivery          35
Shipped - Rejected by Buyer         11
Shipped - Lost in Transit            5
Shipped - Damaged                    1
Name: count, dtype: int64

The dataset contains several order statuses such as **"Shipped"**, **"Cancelled"**, **"Returned"**, and **"Pending"**.

For the sales analysis, only orders corresponding to **completed sales** will be retained.  
Cancelled, returned, or pending orders are excluded.

### Creation of the Sales Dataset

To analyze only actual sales, the dataset is filtered to retain transactions where:

- the quantity sold is strictly positive (**Qty > 0**)
- the order amount is strictly positive (**Amount > 0**)

Cancelled, returned, or uncompleted orders are therefore excluded from the analysis.

In [30]:
df_sales = df_wip[
    (df_wip["Qty"] > 0) &
    (df_wip["Amount"] > 0) &
    (df_wip["Status"].str.startswith("Shipped")) &
    (~df_wip["Status"].str.contains("Returned|Returning|Rejected|Lost|Damaged", na=False))
].copy()
df_sales["Status"].value_counts()

Status
Shipped                         76078
Shipped - Delivered to Buyer    28045
Shipped - Picked Up               945
Shipped - Out for Delivery         35
Name: count, dtype: int64

To analyze actual sales, only orders that have been shipped (statuses starting with **"Shipped"**) are retained.

Cancelled, returned, or pending orders are excluded because they do not correspond to completed sales.

In [31]:
df_sales = df_sales.drop(columns=["Courier Status"])

The **"Courier Status"** column is removed from this dataset because, after filtering the sales, all remaining orders are marked as **"Shipped"**.  
This variable therefore becomes constant and no longer provides useful analytical information.

## 6. Final Dataset

This dataset will be used as the main input for the Excel dashboard built to explore sales performance across categories, regions, and time.
In addition to this final dataset, a cleaned version of the original data ('df_wip') was also exported to facilitate further analysis and validation in Excel.

### Sales Dataset Overview

In [32]:
df_sales.head()

,Order ID,Date,Status,Fulfilment,ship-service-level,Style,SKU,Category,Size,ASIN,Qty,Amount,ship-city,ship-state,ship-postal-code,B2B,promotion_used
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Standard,JNE3781,JNE3781-KR-XXXL,Kurta,3XL,B09K3WFS32,1,406.0,Bengaluru,KARNATAKA,560085,False,True
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Expedited,JNE3371,JNE3371-KR-XL,Kurta,XL,B07WV4JV4D,1,329.0,Navi Mumbai,MAHARASHTRA,410210,True,True
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,1,574.0,Chennai,TAMIL NADU,600073,False,False
5,404-1490984-4578765,2022-04-30,Shipped,Amazon,Expedited,SET264,SET264-KR-NP-XL,Set,XL,B08YN7XDSG,1,824.0,Ghaziabad,UTTAR PRADESH,201102,False,True
6,408-5748499-6859555,2022-04-30,Shipped,Amazon,Expedited,J0095,J0095-SET-L,Set,L,B08CMHNWBN,1,653.0,Chandigarh,CHANDIGARH,160036,False,True


## 7. Final Validation

Before exporting the datasets, several simple checks are performed to ensure the consistency of the produced datasets:

- dataset dimensions;
- absence of missing values in **'Amount'** for **df_sales**;
- absence of zero or negative quantities in **df_sales**.

## 8. Export of the Cleaned Datasets

The cleaned datasets are exported so they can be used in the analysis phase.

- **'amazon_cleaned.csv'**: complete dataset after cleaning columns and data types.
- **'amazon_sales.csv'**: dataset containing only actual sales (shipped orders with positive quantity and amount).

In [33]:
df_wip.to_csv("../data/clean/amazon_cleaned.csv", index=False)
df_sales.to_csv("../data/clean/amazon_sales.csv", index=False)

In [34]:
print("Final validation")

print("\n1. Dataset dimensions")
print(f"- df_wip : {df_wip.shape}")
print(f"- df_sales : {df_sales.shape}")

print("\n2. Missing values in Amount for df_sales")
print(f"- {df_sales['Amount'].isna().sum()}")

print("\n3. Zero or negative quantities in df_sales")
print(f"- {(df_sales['Qty'] <= 0).sum()}")

Final validation

1. Dataset dimensions
- df_wip : (121180, 18)
- df_sales : (105103, 17)

2. Missing values in Amount for df_sales
- 0

3. Zero or negative quantities in df_sales
- 0


## Conclusion

The data was cleaned and prepared for analysis.

The main steps included:
- dataset exploration
- handling missing values
- data type conversion
- feature creation

The resulting datasets will be used to perform sales analysis and build an Excel dashboard.